# 🏗️ Notebook 3: Bad → Best with UML


## 🛠️ Setup

```bash
cd 07-object-oriented-design/uml-basics
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Why draw before refactor?

A quick UML sketch makes **bad structure visible**. When one class has arrows pointing at
half the system, you can *see* that it's doing too much before you touch a single line of code.

We'll take a small e-commerce `Checkout` example through three stages:
1. **Bad** — one God class doing everything.
2. **Better** — split responsibilities by what they do.
3. **Best** — depend on abstractions so each piece is swappable & testable.


## Stage 1 — Bad: the God class

```
┌───────────────────────────────┐
│          Checkout                 │  ← knows EVERYTHING
├────────────────────────────────┤
│ + place_order(cart, user, card)   │
│   - computes total + tax          │
│   - charges credit card (HTTP)    │
│   - writes row to SQL database    │
│   - sends an email                │
└────────────────────────────────┘
```

Smells just from the diagram: **one class with four unrelated reasons to change**
(pricing rule, payment provider, database, email template). Drawing it made that obvious.


In [1]:
# Deliberately bad — do NOT imitate in real code.
class CheckoutBad:
    def place_order(self, cart, user, card):
        # 1. pricing
        total = sum(item['price'] * item['qty'] for item in cart)
        tax = total * 0.08
        grand = total + tax
        # 2. payment (fake HTTP)
        print(f"[http]  POST /charge card={card[-4:]} amount={grand}")
        # 3. persistence (fake SQL)
        print(f"[sql]   INSERT INTO orders (user, total) VALUES ('{user}', {grand})")
        # 4. notification (fake SMTP)
        print(f"[smtp]  to={user}@example.com subject='Thanks for your order'")
        return grand

cart = [{'price': 10, 'qty': 2}, {'price': 5, 'qty': 1}]
CheckoutBad().place_order(cart, 'alice', '4242424242424242')


[http]  POST /charge card=4242 amount=27.0
[sql]   INSERT INTO orders (user, total) VALUES ('alice', 27.0)
[smtp]  to=alice@example.com subject='Thanks for your order'


27.0

### What hurts
- Can't unit-test pricing without mocking HTTP, SQL **and** SMTP.
- Changing the tax rule risks breaking email sending.
- Swapping Stripe for PayPal means editing the same class that stores orders.


## Stage 2 — Better: split by responsibility

Give each reason-to-change its own class. The UML now has small boxes connected by
clear arrows — each arrow is a dependency you can see and reason about.

```
┌─────────────┐     ┌──────────┐
│   Pricing    │◇────▶│  Cart    │
└─────────────┘     └──────────┘
       │                          ┌────────────┐
       ▼                          │ StripePay │
┌─────────────┐ uses    ┌──┼▶           │
│  Checkout    │────────┤  └───────────┘
│ (orchestr.)  │          │  ┌────────────┐
└─────────────┘          │──▶  SqlRepo  │
                           │  └────────────┘
                           │  ┌────────────┐
                           └──▶ EmailNotif│
                              └────────────┘
```


In [2]:
class Pricing:
    TAX = 0.08
    def total(self, cart):
        subtotal = sum(i['price'] * i['qty'] for i in cart)
        return round(subtotal * (1 + self.TAX), 2)

class StripePay:
    def charge(self, card, amount):
        print(f"[stripe] charge card=...{card[-4:]} amount={amount}")

class SqlRepo:
    def save_order(self, user, amount):
        print(f"[sql] INSERT INTO orders VALUES ('{user}', {amount})")

class EmailNotif:
    def thank(self, user):
        print(f"[smtp] to={user}@example.com 'Thanks!'")

class CheckoutBetter:
    def __init__(self):
        self.pricing = Pricing()
        self.payments = StripePay()
        self.orders = SqlRepo()
        self.notifier = EmailNotif()

    def place_order(self, cart, user, card):
        amount = self.pricing.total(cart)
        self.payments.charge(card, amount)
        self.orders.save_order(user, amount)
        self.notifier.thank(user)
        return amount

CheckoutBetter().place_order(cart, 'alice', '4242424242424242')


[stripe] charge card=...4242 amount=27.0
[sql] INSERT INTO orders VALUES ('alice', 27.0)
[smtp] to=alice@example.com 'Thanks!'


27.0

### What improved
- Each class has **one reason to change** (Single Responsibility Principle).
- `Pricing` is a pure function — trivial to unit-test.
- The UML arrows are short and one-directional. No cycles, no God class.

### What still hurts
- `CheckoutBetter` **hard-codes** Stripe, SQL and Email. Swapping to PayPal means editing `Checkout`.
- In tests we can't stub the payment provider without monkey-patching.


## Stage 3 — Best: depend on abstractions

Introduce small **`<<interface>>`** classes. `Checkout` now talks to roles, not vendors.
This is the *Dependency Inversion* idea, and UML makes it obvious: the arrows from
`Checkout` point at interfaces, and the concrete vendors point **up** to the same interfaces.

```
               ┌────────────────┐     ┌─────────────────┐
               │ <<interface>> │     │ <<interface>>   │
               │  PaymentPort  │     │ OrderRepository │
               └──────△───────┘     └──────△─────────┘
                      │                    │
             ┌───────┴───────┐      ┌───┴─────────┐
       StripePay              FakePay   SqlRepo    InMemoryRepo
```


In [3]:
from abc import ABC, abstractmethod

# ── Ports (a.k.a. <<interface>> in UML) ──
class PaymentPort(ABC):
    @abstractmethod
    def charge(self, card: str, amount: float) -> None: ...

class OrderRepository(ABC):
    @abstractmethod
    def save(self, user: str, amount: float) -> int: ...

class Notifier(ABC):
    @abstractmethod
    def thank(self, user: str) -> None: ...

# ── Adapters (real + test doubles) ──
class StripeAdapter(PaymentPort):
    def charge(self, card, amount):
        print(f"[stripe] charge ...{card[-4:]} {amount}")

class FakePay(PaymentPort):
    def __init__(self): self.calls = []
    def charge(self, card, amount): self.calls.append((card, amount))

class InMemoryRepo(OrderRepository):
    def __init__(self): self.rows = []
    def save(self, user, amount):
        self.rows.append((user, amount))
        return len(self.rows)

class PrintNotifier(Notifier):
    def thank(self, user): print(f"[notif] thanks {user}")

# ── Orchestrator depends on interfaces only ──
class CheckoutBest:
    def __init__(self, pricing: Pricing, pay: PaymentPort,
                 repo: OrderRepository, notif: Notifier):
        self.pricing, self.pay, self.repo, self.notif = pricing, pay, repo, notif

    def place_order(self, cart, user, card):
        amount = self.pricing.total(cart)
        self.pay.charge(card, amount)
        order_id = self.repo.save(user, amount)
        self.notif.thank(user)
        return order_id, amount

# Production wiring
prod = CheckoutBest(Pricing(), StripeAdapter(), InMemoryRepo(), PrintNotifier())
print("prod:", prod.place_order(cart, 'alice', '4242424242424242'))


[stripe] charge ...4242 27.0
[notif] thanks alice
prod: (1, 27.0)


### Now tests are trivial

No mocking library needed — just pass the in-memory / fake adapters.


In [4]:
# A real unit test of Checkout without touching Stripe, SQL or SMTP.
fake_pay = FakePay()
repo = InMemoryRepo()
checkout = CheckoutBest(Pricing(), fake_pay, repo, PrintNotifier())

order_id, amount = checkout.place_order(
    cart=[{'price': 100, 'qty': 1}],  # subtotal 100, +8% tax = 108.00
    user='bob',
    card='4000000000000000',
)

assert amount == 108.00, amount
assert fake_pay.calls == [('4000000000000000', 108.00)]
assert repo.rows == [('bob', 108.00)]
assert order_id == 1
print("✅ all assertions passed")


[notif] thanks bob
✅ all assertions passed


## Recap

| Stage | UML tell | Symptom | Fix |
|---|---|---|---|
| Bad | one class, 4 outgoing arrows | untestable, every change risky | split by responsibility |
| Better | concrete classes coupled together | can't swap vendors, hard to fake | introduce `<<interface>>`s |
| Best | arrows point at interfaces | easy tests, swappable adapters | inject dependencies |

The diagrams didn't change the logic — they *revealed* it. That's the real point of UML:
a cheap way to see your design before code solidifies it.
